# 01 - Fink LSST Contract And Schema Exploration

Fixture-based exploration of Swagger contracts and endpoint-specific schema fixtures. This notebook marks inferred field families as reconnaissance, not science-ready data.

In [1]:
from pathlib import Path
import os
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from fink_lsst.contracts import summarize_contracts
from fink_lsst.schema import summarize_schema_fields, summarize_tags, infer_internal_concept_map, internal_data_model_table
from fink_lsst.storage import read_json, safe_artifact_path, write_json

swagger_path = Path("data/fixtures/fink_lsst_swagger.json")
schema_dir = Path("data/fixtures/schema")
tags_path = Path("data/fixtures/fink_lsst_tags.json")
sample_tags_path = Path("data/fixtures/samples/fink_lsst_tags_sample.json")
swagger_path.exists(), schema_dir.exists()

(True, True)

## Swagger Contract Summary

In [2]:
if swagger_path.exists():
    swagger = read_json(swagger_path)
    contracts_df = pd.DataFrame(summarize_contracts(swagger))
    display(contracts_df[["path", "method", "summary", "required_parameters", "optional_parameters"]])
else:
    contracts_df = pd.DataFrame()
    print("No Swagger fixture available yet.")

,path,method,summary,required_parameters,optional_parameters
0,/api/v1/blocks,GET,Retrieve block definition,[],[]
1,/api/v1/conesearch,GET,Retrieve lightcurve data from the Fink/LSST da...,[],"[{'name': 'ra', 'in': 'query', 'required': Fal..."
2,/api/v1/conesearch,POST,Retrieve lightcurve data from the Fink/LSST da...,"[{'name': 'payload', 'in': 'body', 'required':...","[{'name': 'ra', 'in': 'query', 'required': Fal..."
3,/api/v1/cutouts,GET,Retrieve cutout data from the Fink/Rubin datalake,[],"[{'name': 'diaSourceId', 'in': 'query', 'requi..."
4,/api/v1/cutouts,POST,Retrieve cutout data from the Fink/Rubin datalake,"[{'name': 'payload', 'in': 'body', 'required':...","[{'name': 'diaSourceId', 'in': 'query', 'requi..."
5,/api/v1/fp,GET,Retrieve forced photometry data from the Fink/...,[],"[{'name': 'diaObjectId', 'in': 'query', 'requi..."
6,/api/v1/fp,POST,Retrieve forced photometry data from the Fink/...,"[{'name': 'payload', 'in': 'body', 'required':...","[{'name': 'diaObjectId', 'in': 'query', 'requi..."
7,/api/v1/objects,GET,Retrieve object (aggregated) data from the Fin...,[],"[{'name': 'diaObjectId', 'in': 'query', 'requi..."
8,/api/v1/objects,POST,Retrieve object (aggregated) data from the Fin...,"[{'name': 'payload', 'in': 'body', 'required':...","[{'name': 'diaObjectId', 'in': 'query', 'requi..."
9,/api/v1/resolver,GET,None,[],"[{'name': 'resolver', 'in': 'query', 'required..."


## Endpoint Schema Fixtures

In [3]:
schema_frames = []
for path in sorted(schema_dir.glob("fink_lsst_schema_*.json")):
    endpoint = path.stem.removeprefix("fink_lsst_schema_")
    frame = summarize_schema_fields(read_json(path))
    frame["endpoint"] = endpoint
    frame["source_file"] = str(path)
    schema_frames.append(frame)

schema_df = pd.concat(schema_frames, ignore_index=True) if schema_frames else pd.DataFrame()
schema_df.head(40)

,field,family,doc,fink_broker_version,fink_science_version,type,endpoint,source_file
0,apFlux,LSST original fields (r:),Flux in a 12 pixel radius aperture on the diff...,NaN,NaN,"[null, float]",conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
1,apFluxErr,LSST original fields (r:),Estimated uncertainty of apFlux [nJy].,NaN,NaN,"[null, float]",conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
2,apFlux_flag,LSST original fields (r:),General aperture flux algorithm failure flag; ...,NaN,NaN,"[null, boolean]",conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
3,apFlux_flag_apertureTruncated,LSST original fields (r:),Aperture did not fit within measurement image.,NaN,NaN,"[null, boolean]",conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
4,band,LSST original fields (r:),Filter band this source was observed with.,NaN,NaN,"[null, string]",conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
5,bboxSize,LSST original fields (r:),Size of the square bounding box that fully con...,NaN,NaN,int,conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
6,centroid_flag,LSST original fields (r:),General centroid algorithm failure flag; set i...,NaN,NaN,"[null, boolean]",conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
7,clf_cats_class,Fink science module outputs (f:),CATS classifier broad class prediction with th...,4.0,8.26.0,int,conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
8,clf_cats_score,Fink science module outputs (f:),CATS classifier highest probability (0...1). S...,4.1,8.35.0,float,conesearch,data/fixtures/schema/fink_lsst_schema_conesear...
9,clf_earlySNIa_score,Fink science module outputs (f:),Score (0...1) for the early SN Ia classifier (...,4.0,8.26.0,float,conesearch,data/fixtures/schema/fink_lsst_schema_conesear...


## Field Family Map

In [4]:
if not schema_df.empty:
    fields = schema_df["field"].dropna().astype(str).unique().tolist()
    concept_map = infer_internal_concept_map(fields)
    model_table = internal_data_model_table(concept_map)
    model_table["confidence"] = model_table["candidate_field_count"].apply(lambda n: "observed" if n else "uncertain")
    display(model_table)
else:
    concept_map = {}
    model_table = pd.DataFrame()
    print("No endpoint schema fixtures available yet.")

,internal_concept,description,candidate_field_count,candidate_fields,confidence
0,object_id,Object-level identifier fields,2,"diaObjectId, ssObjectId",observed
1,source_id,Alert/source-level identifier fields,6,"diaSourceId, firstDiaSourceMjdTai, firstDiaSou...",observed
2,ra,Right ascension fields,20,"glint_trail, pixelFlags_saturated, pixelFlags_...",observed
3,dec,Declination fields,5,"dec, decErr, ra_dec_Cov, trailDec, trailDecErr",observed
4,time,"Time, MJD, JD, or date fields",8,"firstDiaSourceMjdTai, firstDiaSourceMjdTaiFink...",observed
5,band_or_filter,Filter or band fields,1,band,observed
6,magnitude_or_flux,"Magnitude, flux, uncertainty, or S/N fields",102,"apFlux, apFluxErr, apFlux_flag, apFlux_flag_ap...",observed
7,classification,Classification or model-score fields,5,"clf_cats_class, clf_cats_score, clf_snnSnVsOth...",observed
8,tag,Tag and quality/status fields,0,,uncertain
9,crossmatch_or_context,Catalog crossmatch and contextual fields,9,"xm_gaiadr3_DR3Name, xm_gaiadr3_Plx, xm_gaiadr3...",observed


## Tags

In [5]:
tag_source = sample_tags_path if sample_tags_path.exists() else tags_path
if tag_source.exists():
    tags_df = summarize_tags(read_json(tag_source))
    display(tags_df)
else:
    tags_df = pd.DataFrame()
    print("No tags fixture available yet.")

,tag,API support,description
0,extragalactic_lt20mag_candidate,True,"Select alerts that are rising, bright (mag < 2..."
1,extragalactic_new_candidate,True,Select LSST alerts new (< 5days first appariti...
2,extragalactic_svom,False,Select LSST alerts new (< 5days first appariti...
3,hostless_candidate,True,Select LSST alerts that are hostless according...
4,in_tns,True,Select alerts with a known counterpart in TNS ...
5,most_likely_sn,True,"Selects alerts that are likely to be SN, based..."
6,remove_unlikely_transients,False,Filters out alerts unlikely to be transients o...
7,sn_near_galaxy_candidate,True,Select alerts matching in catalogs to a galaxy...
8,uniform_sample,False,Select 1% of all live alerts in a uniformly ra...


## Save Proposal

In [6]:
proposal = {
    "swagger_fixture": str(swagger_path) if swagger_path.exists() else None,
    "schema_fixtures": [str(path) for path in sorted(schema_dir.glob("fink_lsst_schema_*.json"))],
    "tags_fixture": str(tag_source) if tag_source.exists() else None,
    "concept_map": concept_map,
    "notes": [
        "Generated from Swagger and schema fixtures captured during Checkpoint 0.6.",
        "Fields marked uncertain require real sample rows before science use.",
        "Cutout/image fields are metadata only in this phase; no image data was fetched.",
    ],
}
proposal_path = safe_artifact_path("schema_exploration/internal_data_model_proposal.json", root_name="outputs")
write_json(proposal, proposal_path)
proposal_path

PosixPath('/Users/darim_shamsi/Documents/Fink Analysis/outputs/schema_exploration/internal_data_model_proposal.json')